<a href="https://colab.research.google.com/github/SZ330/EE344-Stock-Market-Predictor/blob/main/SPY_Predictors.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SPY Predictors for 1995 - 2025
This notebook builds and evaluates a machine learning model to predict whether the SPY ETF will go up or down the next trading day based on historical market data. It begins by loading a feature-engineered dataset spanning 1995–2025, removes neutral (no-change) days, and selects a set of technical and calendar-based predictors such as lagged returns, rolling volatility, moving-average ratios, momentum, and volume features. The data is split chronologically into a training period (1995-2010) and a testing period (2010-2025) to preserve the time-series structure and prevent leakage. Three classification models: Logistic Regression, Linear SVM, and Random Forest, are trained (with appropriate scaling for linear models) and evaluated using accuracy, precision, recall, and F1-score. The overall goal of the notebook is to compare these models and determine whether historical price and volume patterns contain predictive signal for next-day market direction, while assessing generalization performance on truly out-of-sample future data.

#### Step 1: Import libraries


In [286]:
import warnings
from itertools import product

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display

from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

warnings.filterwarnings('ignore', category=ConvergenceWarning)
sns.set_theme(style='whitegrid')

RANDOM_STATE = 42


#### Step 2: Load data


In [287]:
# Read the nine-feature SPY dataset
SPY_CSV_PATH = 'SPY_1995_2025_nine_features.csv'
df = pd.read_csv(SPY_CSV_PATH)

# Horizon/target configuration
HORIZON_DAYS = 1
TARGET_COL = f'y_{HORIZON_DAYS}d_dir'
RET_EVAL_COL = f'ret_{HORIZON_DAYS}d_fwd'  # evaluation-only; never a model input

# Basic hygiene
df.columns = df.columns.str.strip()
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

# Keep strictly binary labels
df = df[df[TARGET_COL].isin([-1.0, 1.0])].copy()

print(df[['Date', TARGET_COL, RET_EVAL_COL]].head())
print(df[['Date', TARGET_COL, RET_EVAL_COL]].tail())
print('shape:', df.shape)
print('label distribution:')
print(df[TARGET_COL].value_counts().sort_index())


        Date  y_1d_dir  ret_1d_fwd
0 1995-03-14      -1.0   -0.001892
1 1995-03-15       1.0    0.006000
2 1995-03-16       1.0    0.000994
3 1995-03-20      -1.0   -0.002522
4 1995-03-21       1.0    0.000948
           Date  y_1d_dir  ret_1d_fwd
7709 2025-12-22       1.0    0.004570
7710 2025-12-23       1.0    0.003518
7711 2025-12-24      -1.0   -0.000101
7712 2025-12-26      -1.0   -0.003564
7713 2025-12-29      -1.0   -0.001221
shape: (7714, 17)
label distribution:
y_1d_dir
-1.0    3504
 1.0    4210
Name: count, dtype: int64


#### Step 3: Playground controls
Use this section to shorten the training lookback, limit the backtest span, and compare near-term windows without rerunning the full 1995-2025 walk-forward path.


In [ ]:
# Playground controls for shorter experiments
PLAYGROUND_ENABLED = True

# None uses the latest trading day in the dataset.
# PLAYGROUND_END_DATE = "2007-10-09"
# PLAYGROUND_END_DATE = "2013-12-31"
# PLAYGROUND_END_DATE = "2019-12-31"
# PLAYGROUND_END_DATE = "2021-12-31"
PLAYGROUND_END_DATE = "2024-12-31"

# Retrain each rolling model on only the most recent N calendar years.
PLAYGROUND_TRAIN_YEARS = 3

# Optional static validation buffer used only by the split audit cell below.
PLAYGROUND_VAL_TRADING_DAYS = 0

# Limit the rolling backtest to the last N trading days instead of the full 1995-2025 range.
PLAYGROUND_TEST_TRADING_DAYS = 252

# Advance the rolling cutoff by this many trading days.
PLAYGROUND_ROLL_STEP_DAYS = 1

# Evaluate next-day predictions over short forward windows (trading days).
PLAYGROUND_WINDOW_SPECS = {
    '1D': 1,
    '3D': 3,
    '5D': 5,
    # '10D': 10,
}

DEFAULT_TRAIN_END = pd.to_datetime('2007-09-30')
DEFAULT_VAL_END = pd.to_datetime('2014-12-31')
DEFAULT_WINDOW_SPECS = {'1D': 1, '3D': 3, '5D': 5}
DEFAULT_ROLL_STEP_DAYS = 5


def build_playground_settings(df_src):
    settings = {
        'enabled': False,
        'analysis_df': df_src.copy(),
        'resolved_end_date': df_src['Date'].max(),
        'train_start': df_src['Date'].min(),
        'train_end': DEFAULT_TRAIN_END,
        'val_end': DEFAULT_VAL_END,
        'roll_step_days': DEFAULT_ROLL_STEP_DAYS,
        'window_specs': DEFAULT_WINDOW_SPECS.copy(),
        'rolling_training_mode': 'expanding history',
        'train_years': None,
        'test_trading_days': None,
        'val_trading_days': None,
    }

    if not PLAYGROUND_ENABLED:
        return settings

    if HORIZON_DAYS != 1:
        raise ValueError('The nine-feature dataset currently only supports the 1-day target.')

    if PLAYGROUND_TRAIN_YEARS <= 0:
        raise ValueError('PLAYGROUND_TRAIN_YEARS must be positive.')

    if PLAYGROUND_TEST_TRADING_DAYS <= 0:
        raise ValueError('PLAYGROUND_TEST_TRADING_DAYS must be positive.')

    if PLAYGROUND_VAL_TRADING_DAYS < 0:
        raise ValueError('PLAYGROUND_VAL_TRADING_DAYS cannot be negative.')

    if PLAYGROUND_ROLL_STEP_DAYS <= 0:
        raise ValueError('PLAYGROUND_ROLL_STEP_DAYS must be positive.')

    if not PLAYGROUND_WINDOW_SPECS:
        raise ValueError('PLAYGROUND_WINDOW_SPECS cannot be empty.')

    window_specs = {str(label): int(size) for label, size in PLAYGROUND_WINDOW_SPECS.items()}
    if min(window_specs.values()) <= 0:
        raise ValueError('All playground window sizes must be positive.')

    max_window_days = max(window_specs.values())
    if PLAYGROUND_TEST_TRADING_DAYS < max_window_days:
        raise ValueError(
            'PLAYGROUND_TEST_TRADING_DAYS must be at least as large as the biggest window in '
            f'PLAYGROUND_WINDOW_SPECS ({max_window_days}).'
        )

    resolved_end_date = df_src['Date'].max() if PLAYGROUND_END_DATE is None else pd.to_datetime(PLAYGROUND_END_DATE)
    if pd.isna(resolved_end_date):
        raise ValueError('PLAYGROUND_END_DATE could not be parsed.')

    available_df = df_src[df_src['Date'] <= resolved_end_date].copy().reset_index(drop=True)
    if available_df.empty:
        raise ValueError('No rows are available on or before PLAYGROUND_END_DATE.')

    first_test_start_idx = len(available_df) - PLAYGROUND_TEST_TRADING_DAYS
    if first_test_start_idx <= 0:
        raise ValueError('Not enough rows remain to create both a training slice and a test slice.')

    test_start_date = available_df.loc[first_test_start_idx, 'Date']

    if PLAYGROUND_VAL_TRADING_DAYS > 0:
        val_start_idx = first_test_start_idx - PLAYGROUND_VAL_TRADING_DAYS
        if val_start_idx <= 0:
            raise ValueError('Not enough rows remain to create the requested validation buffer.')
        train_end_date = available_df.loc[val_start_idx, 'Date']
    else:
        train_end_date = test_start_date

    train_start_date = train_end_date - pd.DateOffset(years=PLAYGROUND_TRAIN_YEARS)
    analysis_df = available_df[available_df['Date'] >= train_start_date].copy().reset_index(drop=True)

    train_rows = int((analysis_df['Date'] < train_end_date).sum())
    test_rows = int((analysis_df['Date'] >= test_start_date).sum())

    if train_rows == 0:
        raise ValueError(
            'The requested playground training window is empty. Increase PLAYGROUND_TRAIN_YEARS '
            'or move PLAYGROUND_END_DATE later.'
        )

    if test_rows < max_window_days:
        raise ValueError('The requested playground test slice is too short for the configured window sizes.')

    settings.update({
        'enabled': True,
        'analysis_df': analysis_df,
        'resolved_end_date': analysis_df['Date'].max(),
        'train_start': analysis_df.loc[analysis_df['Date'] < train_end_date, 'Date'].min(),
        'train_end': train_end_date,
        'val_end': test_start_date,
        'roll_step_days': PLAYGROUND_ROLL_STEP_DAYS,
        'window_specs': window_specs,
        'rolling_training_mode': f'fixed {PLAYGROUND_TRAIN_YEARS}-year lookback',
        'train_years': PLAYGROUND_TRAIN_YEARS,
        'test_trading_days': PLAYGROUND_TEST_TRADING_DAYS,
        'val_trading_days': PLAYGROUND_VAL_TRADING_DAYS,
    })

    playground_audit = pd.DataFrame({
        'Setting': [
            'Playground mode',
            'Anchored end date',
            'Training lookback',
            'Validation buffer (trading days)',
            'Test span (trading days)',
            'Rolling step (trading days)',
            'Window sizes',
            'Active data slice',
        ],
        'Value': [
            'enabled',
            settings['resolved_end_date'].date(),
            f"{PLAYGROUND_TRAIN_YEARS} calendar years",
            PLAYGROUND_VAL_TRADING_DAYS,
            PLAYGROUND_TEST_TRADING_DAYS,
            PLAYGROUND_ROLL_STEP_DAYS,
            window_specs,
            f"{analysis_df['Date'].min().date()} to {analysis_df['Date'].max().date()} ({len(analysis_df):,} rows)",
        ],
    })

    display(playground_audit)
    return settings


PLAYGROUND_SETTINGS = build_playground_settings(df)
ANALYSIS_DF = PLAYGROUND_SETTINGS['analysis_df']
TRAIN_END = PLAYGROUND_SETTINGS['train_end']
VAL_END = PLAYGROUND_SETTINGS['val_end']
ROLL_STEP_DAYS = PLAYGROUND_SETTINGS['roll_step_days']
WINDOW_SPECS = PLAYGROUND_SETTINGS['window_specs']
WINDOW_DISPLAY_ORDER = list(WINDOW_SPECS)
MAX_WINDOW_DAYS = max(WINDOW_SPECS.values())

if not PLAYGROUND_SETTINGS['enabled']:
    print('Playground mode is off. The notebook will use the original long-history split and window settings.')


,Setting,Value
0,Playground mode,enabled
1,Anchored end date,2024-12-31
2,Training lookback,3 calendar years
3,Validation buffer (trading days),0
4,Test span (trading days),252
5,Rolling step (trading days),1
6,Window sizes,"{'1D': 1, '3D': 3, '5D': 5}"
7,Active data slice,"2021-01-04 to 2024-12-31 (1,003 rows)"


#### Step 4: Feature audit


In [289]:
# Main nine-feature daily input set
BASE_SPY_FEATURE_COLS = [
    'rsi_14',
    'sma_50',
    'adx_20',
    'volume',
    'corr_24',
    'prev_open_close',
    'prev_close_high',
    'prev_close_low',
    'momentum_20',
]

FEATURE_COLS = [c for c in BASE_SPY_FEATURE_COLS if c in df.columns]
CAT_COLS = []

missing = sorted(set(BASE_SPY_FEATURE_COLS) - set(FEATURE_COLS))
assert not missing, f'Missing expected SPY features: {missing}'
assert FEATURE_COLS == BASE_SPY_FEATURE_COLS, 'The main feature set no longer matches the intended nine daily features'
assert TARGET_COL in df.columns, f'Missing target column: {TARGET_COL}'
assert RET_EVAL_COL in df.columns, f'Missing evaluation column: {RET_EVAL_COL}'
assert TARGET_COL not in FEATURE_COLS, 'Leakage: target found in features'
assert RET_EVAL_COL not in FEATURE_COLS, 'Leakage: forward return found in features'

print('Main feature set audit:')
print('Feature names:', FEATURE_COLS)
print('Number of features:', len(FEATURE_COLS))
print('Using intended daily-feature set:', FEATURE_COLS == BASE_SPY_FEATURE_COLS)


Main feature set audit:
Feature names: ['rsi_14', 'sma_50', 'adx_20', 'volume', 'corr_24', 'prev_open_close', 'prev_close_high', 'prev_close_low', 'momentum_20']
Number of features: 9
Using intended daily-feature set: True


#### Step 5: Existing split audit


In [290]:
def build_scenario(df_src, feature_cols, name):
    d = df_src.copy()
    d = d.dropna(subset=feature_cols + [TARGET_COL, RET_EVAL_COL]).copy()

    use_validation = VAL_END > TRAIN_END

    train_df = d[d['Date'] < TRAIN_END].copy()
    if use_validation:
        val_df = d[(d['Date'] >= TRAIN_END) & (d['Date'] < VAL_END)].copy()
        test_df = d[d['Date'] >= VAL_END].copy()
    else:
        val_df = d.iloc[0:0].copy()
        test_df = d[d['Date'] >= TRAIN_END].copy()

    assert len(train_df) > 0, f'{name}: Train split is empty'
    assert len(test_df) > 0, f'{name}: Test split is empty'
    if use_validation:
        assert len(val_df) > 0, f'{name}: Val split is empty'

    return {
        'name': name,
        'feature_cols': feature_cols,
        'df': d,
        'train_df': train_df,
        'val_df': val_df,
        'test_df': test_df,
        'X_train': train_df[feature_cols],
        'y_train': train_df[TARGET_COL],
        'X_val': val_df[feature_cols],
        'y_val': val_df[TARGET_COL],
        'X_test': test_df[feature_cols],
        'y_test': test_df[TARGET_COL],
        'ret_train': train_df[RET_EVAL_COL],
        'ret_val': val_df[RET_EVAL_COL],
        'ret_test': test_df[RET_EVAL_COL],
    }


SPY_SCENARIO = build_scenario(ANALYSIS_DF, FEATURE_COLS, 'SPY only')
SCENARIOS = {'SPY only': SPY_SCENARIO}
ACTIVE_SCENARIO = 'SPY only'

X_train = SPY_SCENARIO['X_train']
y_train = SPY_SCENARIO['y_train']
X_val = SPY_SCENARIO['X_val']
y_val = SPY_SCENARIO['y_val']
X_test = SPY_SCENARIO['X_test']
y_test = SPY_SCENARIO['y_test']
train_df = SPY_SCENARIO['train_df']
val_df = SPY_SCENARIO['val_df']
test_df = SPY_SCENARIO['test_df']

print('Active main scenario:', ACTIVE_SCENARIO)
print('Split logic audit:')
print('Mode:', 'playground' if PLAYGROUND_SETTINGS['enabled'] else 'legacy long-history split')
print('TRAIN: Date <', TRAIN_END.date())
if len(val_df) > 0:
    print('VAL:   Date >=', TRAIN_END.date(), 'and Date <', VAL_END.date())
    print('TEST:  Date >=', VAL_END.date())
else:
    print('VAL:   skipped (PLAYGROUND_VAL_TRADING_DAYS = 0)')
    print('TEST:  Date >=', TRAIN_END.date())
print()
print('ACTIVE RANGE:', SPY_SCENARIO['df']['Date'].min(), 'to', SPY_SCENARIO['df']['Date'].max(), f"({len(SPY_SCENARIO['df']):,})")
print('TRAIN:', train_df['Date'].min(), 'to', train_df['Date'].max(), f"({len(train_df):,})")
if len(val_df) > 0:
    print('VAL:  ', val_df['Date'].min(), 'to', val_df['Date'].max(), f"({len(val_df):,})")
else:
    print('VAL:   skipped (0 rows)')
print('TEST: ', test_df['Date'].min(), 'to', test_df['Date'].max(), f"({len(test_df):,})")
print('Num main-path features:', len(SPY_SCENARIO['feature_cols']))


Active main scenario: SPY only
Split logic audit:
Mode: playground
TRAIN: Date < 2024-01-02
VAL:   skipped (PLAYGROUND_VAL_TRADING_DAYS = 0)
TEST:  Date >= 2024-01-02

ACTIVE RANGE: 2021-01-04 00:00:00 to 2024-12-31 00:00:00 (1,003)
TRAIN: 2021-01-04 00:00:00 to 2023-12-29 00:00:00 (751)
VAL:   skipped (0 rows)
TEST:  2024-01-02 00:00:00 to 2024-12-31 00:00:00 (252)
Num main-path features: 9


#### Step 6: Reusable evaluation helpers


In [291]:
def extract_positive_class_scores(model, X):
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
        classes = list(model.classes_)
        pos_idx = classes.index(1.0)
        return proba[:, pos_idx]

    if hasattr(model, 'decision_function'):
        scores = model.decision_function(X)
        if np.ndim(scores) > 1:
            scores = scores[:, 0]
        return scores

    return None



def evaluate_classifier(model, X, y_true):
    y_true_array = pd.Series(y_true).to_numpy()
    y_pred = model.predict(X)
    scores = extract_positive_class_scores(model, X)

    roc_auc = np.nan
    if scores is not None:
        try:
            roc_auc = roc_auc_score((y_true_array == 1.0).astype(int), scores)
        except ValueError:
            roc_auc = np.nan

    return {
        'accuracy': accuracy_score(y_true_array, y_pred),
        'precision': precision_score(y_true_array, y_pred, pos_label=1.0, zero_division=0),
        'recall': recall_score(y_true_array, y_pred, pos_label=1.0, zero_division=0),
        'f1': f1_score(y_true_array, y_pred, pos_label=1.0, zero_division=0),
        'roc_auc': roc_auc,
        'confusion_matrix': confusion_matrix(y_true_array, y_pred, labels=[-1.0, 1.0]),
        'predictions': y_pred,
        'scores': scores,
    }



def metric_sort_key(metric_dict):
    roc_auc = metric_dict['roc_auc']
    safe_auc = -np.inf if pd.isna(roc_auc) else roc_auc
    return (metric_dict['accuracy'], metric_dict['f1'], safe_auc)


#### Step 7: Model definitions


In [292]:
MODEL_CONFIGS = {
    'SVM': {'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale'},
    'Decision Tree': {'criterion': 'gini', 'min_samples_split': 4},
    'Logistic Regression': {'solver': 'lbfgs', 'max_iter': 100, 'C': 1.0},
    'Naive Bayes': {},
    'KNN': {'n_neighbors': 20, 'metric': 'minkowski'},
    'Random Forest': {'n_estimators': 80, 'criterion': 'gini', 'min_samples_leaf': 4},
    'ANN': {'hidden_layer_sizes': (100,), 'activation': 'relu', 'solver': 'adam', 'max_iter': 200},
}

SCALED_MODELS = {'SVM', 'Logistic Regression', 'Naive Bayes', 'KNN', 'ANN'}



def build_estimator(model_name, params):
    if model_name == 'SVM':
        return SVC(
            C=params['C'],
            kernel=params['kernel'],
            degree=params.get('degree', 3),
            gamma=params['gamma'],
        )

    if model_name == 'Decision Tree':
        return DecisionTreeClassifier(
            criterion=params['criterion'],
            min_samples_split=params['min_samples_split'],
            max_depth=params.get('max_depth'),
            random_state=RANDOM_STATE,
        )

    if model_name == 'Logistic Regression':
        return LogisticRegression(
            C=params['C'],
            solver=params['solver'],
            max_iter=params['max_iter'],
        )

    if model_name == 'Naive Bayes':
        return GaussianNB()

    if model_name == 'KNN':
        return KNeighborsClassifier(
            n_neighbors=params['n_neighbors'],
            leaf_size=params.get('leaf_size', 30),
            metric=params['metric'],
        )

    if model_name == 'Random Forest':
        return RandomForestClassifier(
            n_estimators=params['n_estimators'],
            criterion=params['criterion'],
            min_samples_leaf=params['min_samples_leaf'],
            max_depth=params.get('max_depth'),
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )

    if model_name == 'ANN':
        return MLPClassifier(
            hidden_layer_sizes=params['hidden_layer_sizes'],
            activation=params['activation'],
            solver=params['solver'],
            max_iter=params['max_iter'],
            shuffle=False,
            random_state=RANDOM_STATE,
        )

    raise ValueError(f'Unsupported model: {model_name}')



def build_model_pipeline(model_name, params):
    steps = []

    if model_name in SCALED_MODELS:
        steps.append(('scaler', MinMaxScaler()))

    steps.append(('model', build_estimator(model_name, params)))
    return Pipeline(steps)


model_definition_audit = pd.DataFrame({
    'Model': list(MODEL_CONFIGS.keys()),
    'Fixed Params': [MODEL_CONFIGS[model_name] for model_name in MODEL_CONFIGS],
    'Uses MinMaxScaler': [model_name in SCALED_MODELS for model_name in MODEL_CONFIGS],
})

print('Main comparison is SPY-only and paper-inspired, not a Tesla replication.')
display(model_definition_audit)


Main comparison is SPY-only and paper-inspired, not a Tesla replication.


,Model,Fixed Params,Uses MinMaxScaler
0,SVM,"{'C': 1.0, 'kernel': 'rbf', 'gamma': 'scale'}",True
1,Decision Tree,"{'criterion': 'gini', 'min_samples_split': 4}",False
2,Logistic Regression,"{'solver': 'lbfgs', 'max_iter': 100, 'C': 1.0}",True
3,Naive Bayes,{},True
4,KNN,"{'n_neighbors': 20, 'metric': 'minkowski'}",True
5,Random Forest,"{'n_estimators': 80, 'criterion': 'gini', 'min...",False
6,ANN,"{'hidden_layer_sizes': (100,), 'activation': '...",True


#### Step 8: Rolling evaluation setup


In [293]:
rolling_df = pd.concat(
    [SPY_SCENARIO['train_df'], SPY_SCENARIO['test_df']],
    ignore_index=True,
).reset_index(drop=True)

assert len(SPY_SCENARIO['train_df']) > 0, 'Rolling evaluation requires a non-empty training slice'
assert len(SPY_SCENARIO['test_df']) >= MAX_WINDOW_DAYS, (
    'Rolling evaluation requires the test slice to be at least as long as the largest configured window'
)

INITIAL_TRAIN_CUTOFF_IDX = len(SPY_SCENARIO['train_df']) - 1
INITIAL_TRAIN_CUTOFF_DATE = rolling_df.loc[INITIAL_TRAIN_CUTOFF_IDX, 'Date']
TEST_START = SPY_SCENARIO['test_df']['Date'].min()
TEST_END = SPY_SCENARIO['test_df']['Date'].max()

window_metric_stores = {
    window_label: {model_name: [] for model_name in MODEL_CONFIGS}
    for window_label in WINDOW_DISPLAY_ORDER
}

TOTAL_ROLLING_ITERATIONS = 0
current_cutoff_idx = INITIAL_TRAIN_CUTOFF_IDX

while current_cutoff_idx + MAX_WINDOW_DAYS < len(rolling_df):
    TOTAL_ROLLING_ITERATIONS += 1
    current_cutoff_idx += ROLL_STEP_DAYS

assert TOTAL_ROLLING_ITERATIONS > 0, 'No rolling iterations fit inside the configured test span.'

rolling_setup_df = pd.DataFrame({
    'Setting': [
        'Mode',
        'Rolling training mode',
        'First train cutoff in data',
        'TEST_START',
        'TEST_END',
        'Rolling step (trading days)',
        'Window sizes',
        'Total rolling iterations',
    ],
    'Value': [
        'playground' if PLAYGROUND_SETTINGS['enabled'] else 'legacy',
        PLAYGROUND_SETTINGS['rolling_training_mode'],
        INITIAL_TRAIN_CUTOFF_DATE.date(),
        TEST_START.date(),
        TEST_END.date(),
        ROLL_STEP_DAYS,
        WINDOW_SPECS,
        TOTAL_ROLLING_ITERATIONS,
    ],
})

print('Validation split is intentionally unused in the rolling path below.')
print('Forward windows begin immediately after each training cutoff.')
display(rolling_setup_df)


Validation split is intentionally unused in the rolling path below.
Forward windows begin immediately after each training cutoff.


,Setting,Value
0,Mode,playground
1,Rolling training mode,fixed 3-year lookback
2,First train cutoff in data,2023-12-29
3,TEST_START,2024-01-02
4,TEST_END,2024-12-31
5,Rolling step (trading days),1
6,Window sizes,"{'1D': 1, '3D': 3, '5D': 5}"
7,Total rolling iterations,248


#### Step 9: Rolling training loop


In [294]:
rolling_metric_rows = []
rolling_iteration_rows = []
current_cutoff_idx = INITIAL_TRAIN_CUTOFF_IDX
iteration_number = 0

while current_cutoff_idx + MAX_WINDOW_DAYS < len(rolling_df):
    iteration_number += 1
    train_cutoff_date = rolling_df.loc[current_cutoff_idx, 'Date']

    if PLAYGROUND_SETTINGS['enabled']:
        train_window_start = train_cutoff_date - pd.DateOffset(years=PLAYGROUND_SETTINGS['train_years'])
        train_slice = rolling_df[
            (rolling_df['Date'] >= train_window_start) & (rolling_df['Date'] <= train_cutoff_date)
        ].copy()
    else:
        train_slice = rolling_df.iloc[: current_cutoff_idx + 1].copy()

    X_train_roll = train_slice[FEATURE_COLS]
    y_train_roll = train_slice[TARGET_COL]

    if y_train_roll.nunique() < 2:
        raise ValueError(
            'A rolling training window collapsed to one class. Increase PLAYGROUND_TRAIN_YEARS '
            'or choose a different PLAYGROUND_END_DATE.'
        )

    iteration_summary = {
        'Iteration': iteration_number,
        'Train Start Date': train_slice['Date'].iloc[0],
        'Train Cutoff Date': train_cutoff_date,
        'Train Rows': len(train_slice),
    }

    for window_label, window_size in WINDOW_SPECS.items():
        window_slice = rolling_df.iloc[current_cutoff_idx + 1 : current_cutoff_idx + 1 + window_size].copy()
        iteration_summary[f'{window_label} Start'] = window_slice['Date'].iloc[0]
        iteration_summary[f'{window_label} End'] = window_slice['Date'].iloc[-1]
        iteration_summary[f'{window_label} Rows'] = len(window_slice)

    rolling_iteration_rows.append(iteration_summary)

    for model_name, params in MODEL_CONFIGS.items():
        model = build_model_pipeline(model_name, params)
        model.fit(X_train_roll, y_train_roll)

        for window_label, window_size in WINDOW_SPECS.items():
            window_slice = rolling_df.iloc[current_cutoff_idx + 1 : current_cutoff_idx + 1 + window_size].copy()
            X_window = window_slice[FEATURE_COLS]
            y_window = window_slice[TARGET_COL]
            metrics = evaluate_classifier(model, X_window, y_window)

            metric_record = {
                'Iteration': iteration_number,
                'Train Start Date': train_slice['Date'].iloc[0],
                'Train Cutoff Date': train_cutoff_date,
                'Test Start Date': window_slice['Date'].iloc[0],
                'Test End Date': window_slice['Date'].iloc[-1],
                'Train Rows': len(train_slice),
                'Test Rows': len(window_slice),
                'Accuracy': metrics['accuracy'],
                'Precision': metrics['precision'],
                'Recall': metrics['recall'],
                'F1': metrics['f1'],
                'ROC-AUC': metrics['roc_auc'],
            }

            window_metric_stores[window_label][model_name].append(metric_record)
            rolling_metric_rows.append({
                'Model': model_name,
                'Window': window_label,
                **metric_record,
            })

    if iteration_number % 50 == 0 or iteration_number == TOTAL_ROLLING_ITERATIONS:
        print(
            f'Completed {iteration_number}/{TOTAL_ROLLING_ITERATIONS} rolling iterations '
            f'(latest train cutoff: {train_cutoff_date.date()})'
        )

    current_cutoff_idx += ROLL_STEP_DAYS

rolling_metrics_long = pd.DataFrame(rolling_metric_rows)
rolling_iteration_schedule = pd.DataFrame(rolling_iteration_rows)

print('Rolling loop complete. Models are retrained from scratch at each cutoff.')
print('Total metric records collected:', len(rolling_metrics_long))


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-pac

Completed 50/248 rolling iterations (latest train cutoff: 2024-03-12)


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-pac

Completed 100/248 rolling iterations (latest train cutoff: 2024-05-22)


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-pac

Completed 150/248 rolling iterations (latest train cutoff: 2024-08-05)


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-pac

Completed 200/248 rolling iterations (latest train cutoff: 2024-10-15)


/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/Library/Frameworks/Python.framework/Versions/3.9/lib/python3.9/site-pac

Completed 248/248 rolling iterations (latest train cutoff: 2024-12-23)
Rolling loop complete. Models are retrained from scratch at each cutoff.
Total metric records collected: 5208


#### Step 10: Metric aggregation


In [295]:
summary_rows = []

for window_label in WINDOW_DISPLAY_ORDER:
    metric_store = window_metric_stores[window_label]
    for model_name, metric_records in metric_store.items():
        metric_frame = pd.DataFrame(metric_records)
        summary_rows.append({
            'Model': model_name,
            'Window': window_label,
            'Average Accuracy': metric_frame['Accuracy'].mean(),
            'Average Precision': metric_frame['Precision'].mean(),
            'Average Recall': metric_frame['Recall'].mean(),
            'Average F1': metric_frame['F1'].mean(),
            'Average ROC-AUC': metric_frame['ROC-AUC'].mean(),
            'Iterations': len(metric_frame),
        })

results_table = pd.DataFrame(summary_rows)
results_table['Window'] = pd.Categorical(
    results_table['Window'],
    categories=WINDOW_DISPLAY_ORDER,
    ordered=True,
)
results_table = results_table.sort_values(
    ['Window', 'Average Accuracy'],
    ascending=[True, False],
).reset_index(drop=True)

rolling_coverage_table = (
    rolling_metrics_long.groupby('Window')
    .agg(
        Iterations=('Iteration', 'nunique'),
        First_Test_Start=('Test Start Date', 'min'),
        Last_Test_End=('Test End Date', 'max'),
    )
    .reindex(WINDOW_DISPLAY_ORDER)
    .reset_index()
)

print('Average metrics are aggregated across all rolling iterations for each model-window pair.')
display(rolling_coverage_table)


Average metrics are aggregated across all rolling iterations for each model-window pair.


,Window,Iterations,First_Test_Start,Last_Test_End
0,1D,248,2024-01-02,2024-12-24
1,3D,248,2024-01-02,2024-12-27
2,5D,248,2024-01-02,2024-12-31


#### Step 11: Final results table


In [296]:
results_table_display = results_table.copy()
results_table_display['Window'] = results_table_display['Window'].astype(str)
metric_cols = [
    'Average Accuracy',
    'Average Precision',
    'Average Recall',
    'Average F1',
    'Average ROC-AUC',
]
results_table_display[metric_cols] = results_table_display[metric_cols].round(4)

print(
    'Short-term forward predictive performance averaged across rolling post-training windows '
    f'({TOTAL_ROLLING_ITERATIONS} iterations, {ROLL_STEP_DAYS}-trading-day increments).'
)
display(
    results_table_display[
        [
            'Model',
            'Window',
            'Average Accuracy',
            'Average Precision',
            'Average Recall',
            'Average F1',
            'Average ROC-AUC',
        ]
    ]
)


Short-term forward predictive performance averaged across rolling post-training windows (248 iterations, 1-trading-day increments).


,Model,Window,Average Accuracy,Average Precision,Average Recall,Average F1,Average ROC-AUC
0,Logistic Regression,1D,0.5968,0.5726,0.5726,0.5726,NaN
1,Naive Bayes,1D,0.5847,0.5403,0.5403,0.5403,NaN
2,ANN,1D,0.5847,0.5847,0.5847,0.5847,NaN
3,SVM,1D,0.5685,0.5403,0.5403,0.5403,NaN
4,Random Forest,1D,0.5645,0.4355,0.4355,0.4355,NaN
5,KNN,1D,0.5605,0.4274,0.4274,0.4274,NaN
6,Decision Tree,1D,0.4839,0.3185,0.3185,0.3185,NaN
7,Logistic Regression,3D,0.5927,0.5739,0.8965,0.6718,0.2926
8,ANN,3D,0.5874,0.5874,0.9274,0.6915,0.3097
9,Naive Bayes,3D,0.5820,0.5403,0.8508,0.6355,0.3892


#### Step 12: Optional legacy sections
The main notebook path above is intentionally limited to the SPY-only daily classifier comparison. Threshold sweeps, constrained-threshold logic, strategy-return reporting, extra baselines, deeper models, and the VIX extension are kept out of the main path.


In [297]:
RUN_OPTIONAL_VIX_EXTENSION = False
RUN_OPTIONAL_STRATEGY_ANALYSIS = False
RUN_LEGACY_EXPERIMENTS = False



def build_optional_vix_scenario():
    vix_df = pd.read_csv('VIX_1995_to_2025_feature_expansion_v5.csv')
    vix_df.columns = vix_df.columns.str.strip()
    vix_df['Date'] = pd.to_datetime(vix_df['Date'], errors='coerce')
    vix_df = vix_df.dropna(subset=['Date']).sort_values('Date').reset_index(drop=True)

    vix_feature_cols_raw = [c for c in vix_df.columns if c != 'Date']
    vix_df = vix_df.rename(columns={c: f'vx_{c}' for c in vix_feature_cols_raw})
    vix_feature_cols = [f'vx_{c}' for c in vix_feature_cols_raw]

    df_with_vix = df.merge(vix_df, on='Date', how='left')
    return build_scenario(df_with_vix, FEATURE_COLS + vix_feature_cols, 'SPY + VIX')


if RUN_OPTIONAL_VIX_EXTENSION:
    optional_vix_scenario = build_optional_vix_scenario()
    print('Optional VIX extension ready with the same split logic.')
    print('Optional VIX feature count:', len(optional_vix_scenario['feature_cols']))
else:
    optional_vix_scenario = None
    print('Optional VIX extension is disabled in the main notebook path.')

print('Optional strategy analysis enabled:', RUN_OPTIONAL_STRATEGY_ANALYSIS)
print('Legacy experiments enabled:', RUN_LEGACY_EXPERIMENTS)


Optional VIX extension is disabled in the main notebook path.
Optional strategy analysis enabled: False
Legacy experiments enabled: False
